# Internship data exploration and quality analysis

This notebook is for observation only. It keeps the source workbook in `df_raw`; exploratory calculations use derived objects and do not normalize, remove, or impute source values.

## 1. Imports and configuration

Import the initial analysis libraries and resolve the project root from either the project root or the notebook folder.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    project_root = cwd.parent
elif (cwd / "notebooks").is_dir() or (cwd / "data").is_dir():
    project_root = cwd
elif (cwd.parent / "notebooks").is_dir() or (cwd.parent / "data").is_dir():
    project_root = cwd.parent
else:
    project_root = cwd

excel_path = project_root / "data" / "raw" / "ExportStagesentreprisessujets2020-2027.xlsx"
print(f"Project root: {project_root}")
print(f"Source workbook: {excel_path}")

## 2. Load the Excel dataset

Load the provided workbook without changing it. If it is absent, the error identifies the required location.

In [ ]:
if not excel_path.is_file():
    raise FileNotFoundError(
        f"Excel dataset not found at {excel_path}. "
        "Place the source workbook in data/raw/ and rerun this cell."
    )

df_raw = pd.read_excel(excel_path)
print("Workbook loaded into df_raw; source values have not been modified.")

## 3. Basic dataset overview

Inspect the dimensions, sample records, and high-level descriptive statistics.

In [ ]:
print(f"Rows: {df_raw.shape[0]:,} | Columns: {df_raw.shape[1]:,}")
display(df_raw.head())
display(df_raw.describe(include="all").T)

## 4. Column inspection

Review column names, non-null counts, and example values before making assumptions about the schema.

In [ ]:
column_inspection = pd.DataFrame({
    "column": df_raw.columns.astype(str),
    "non_null_count": df_raw.notna().sum().values,
    "example_values": [df_raw[column].dropna().head(3).tolist() for column in df_raw.columns],
})
display(column_inspection)

## 5. Data types

Compare pandas' inferred types and count distinct non-null values for each column.

In [ ]:
display(pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "non_null_count": df_raw.notna().sum(),
    "unique_non_null": df_raw.nunique(dropna=True),
}))

## 6. Missing values

Quantify missing cells and their share by column. No values are filled or changed.

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_percent": df_raw.isna().mean().mul(100),
}).sort_values("missing_count", ascending=False)
display(missing_summary)

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_percent": df_raw.isna().mean().mul(100),
}).sort_values("missing_count", ascending=False)
display(missing_summary)

## 7. Duplicate records

Count exact duplicate rows as an observation only; do not remove them.

In [ ]:
exact_duplicate_count = int(df_raw.duplicated().sum())
print(f"Exact duplicate rows (excluding each first occurrence): {exact_duplicate_count:,}")
display(df_raw.loc[df_raw.duplicated(keep=False)].head(10))

## 8. Unique values and categorical distributions

Show a small sample of values per column and distributions for object/category columns with manageable cardinality.

In [ ]:
for column in df_raw.columns:
    print(f"{column}: {df_raw[column].dropna().unique()[:10].tolist()}")

categorical_columns = df_raw.select_dtypes(include=["object", "category"]).columns
for column in categorical_columns:
    if df_raw[column].nunique(dropna=True) <= 30:
        print(f"\n{column} (top 20 values):")
        display(df_raw[column].value_counts(dropna=False).head(20).rename("count").to_frame())

## 9. Date analysis

Identify columns whose names suggest dates, then inspect parseable date ranges on a temporary Series. This does not update `df_raw`.

In [ ]:
date_columns = [column for column in df_raw.columns if "date" in str(column).casefold()]
print(f"Columns with 'date' in their name: {date_columns}")
for column in date_columns:
    parsed_dates = pd.to_datetime(df_raw[column], errors="coerce")
    print(
        f"{column}: parseable={parsed_dates.notna().sum():,}, "
        f"min={parsed_dates.min()}, max={parsed_dates.max()}"
    )

## 10. Duration analysis

Inspect columns with duration-like names and numeric summaries where the dtype is already numeric; units and meaning must be confirmed from the data context.

In [ ]:
duration_columns = [
    column for column in df_raw.columns
    if any(term in str(column).casefold() for term in ("duration", "durée", "duree", "period", "période", "periode"))
]
print(f"Columns with duration/period-like names: {duration_columns}")
for column in duration_columns:
    if pd.api.types.is_numeric_dtype(df_raw[column]):
        display(df_raw[column].describe().rename(column).to_frame())
    else:
        display(df_raw[column].value_counts(dropna=False).head(20).rename("count").to_frame())

## 11. Identifier analysis

Profile columns whose names suggest identifiers. Uniqueness and missingness are descriptive signals, not rules about validity.

In [ ]:
identifier_columns = [
    column for column in df_raw.columns
    if any(term in str(column).casefold() for term in ("id", "identifier", "identifiant", "code", "num", "réf", "ref"))
]
identifier_summary = pd.DataFrame({
    "non_null": df_raw[identifier_columns].notna().sum(),
    "unique_non_null": df_raw[identifier_columns].nunique(dropna=True),
    "missing": df_raw[identifier_columns].isna().sum(),
})
print(f"Columns selected by identifier-like names: {identifier_columns}")
display(identifier_summary)

## 12. Company analysis

Review likely company-related columns only when suggested by their names; confirm these candidates against the workbook documentation.

In [ ]:
company_columns = [
    column for column in df_raw.columns
    if any(term in str(column).casefold() for term in ("company", "entreprise", "société", "societe", "employeur"))
]
print(f"Company-like columns: {company_columns}")
for column in company_columns:
    display(df_raw[column].value_counts(dropna=False).head(20).rename("count").to_frame())

## 13. Country and city analysis

Summarize likely country and city columns without normalizing spelling, accents, or casing.

In [ ]:
location_columns = [
    column for column in df_raw.columns
    if any(term in str(column).casefold() for term in ("country", "pays", "city", "ville"))
]
print(f"Country/city-like columns: {location_columns}")
for column in location_columns:
    display(df_raw[column].value_counts(dropna=False).head(20).rename("count").to_frame())

## 14. Stage title analysis

Inspect likely title or subject columns using their names, while keeping all source strings unchanged.

In [ ]:
title_columns = [
    column for column in df_raw.columns
    if any(term in str(column).casefold() for term in ("title", "titre", "sujet", "intitulé", "intitule"))
]
print(f"Title/subject-like columns: {title_columns}")
for column in title_columns:
    display(df_raw[column].dropna().head(20).to_frame())

## 15. Stage description analysis

Inspect likely description columns and a small sample to understand their content and structure.

In [ ]:
description_columns = [
    column for column in df_raw.columns
    if any(term in str(column).casefold() for term in ("description", "descriptif", "mission", "detail", "détail"))
]
print(f"Description-like columns: {description_columns}")
for column in description_columns:
    display(df_raw[column].dropna().head(5).to_frame())

## 16. Text length analysis

Measure lengths on temporary string views of likely text columns. This is only a profiling step; source values remain unchanged.

In [ ]:
text_columns = list(dict.fromkeys(title_columns + description_columns))
for column in text_columns:
    text_lengths = df_raw[column].dropna().astype(str).str.len()
    print(f"{column} character lengths:")
    display(text_lengths.describe().rename("characters").to_frame())
    if not text_lengths.empty:
        sns.histplot(text_lengths, bins=30)
        plt.title(f"Text length distribution: {column}")
        plt.xlabel("Character count")
        plt.show()

## 17. Data quality checks

Assemble neutral checks for empty columns, missingness, and exact duplicated rows. These are observations, not automatic cleaning actions.

In [ ]:
quality_checks = pd.DataFrame({
    "row_count": [len(df_raw)],
    "column_count": [df_raw.shape[1]],
    "fully_empty_columns": [int(df_raw.isna().all().sum())],
    "rows_with_any_missing": [int(df_raw.isna().any(axis=1).sum())],
    "exact_duplicate_rows": [int(df_raw.duplicated().sum())],
})
display(quality_checks)

## 18. Potential anomalies

Display numeric range summaries and potential IQR outlier counts as prompts for review. Statistical flags are not proof of invalid data.

In [ ]:
numeric_columns = df_raw.select_dtypes(include="number").columns
anomaly_rows = []
for column in numeric_columns:
    values = df_raw[column].dropna()
    if values.empty:
        continue
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    anomaly_rows.append({
        "column": column,
        "min": values.min(),
        "max": values.max(),
        "potential_iqr_outliers": int(((values < lower_bound) | (values > upper_bound)).sum()),
    })
display(pd.DataFrame(anomaly_rows))

## 19. Cleaning decisions

Record evidence and questions here after reviewing the results. Do not apply cleaning decisions in this exploration notebook.

## 20. Summary and next steps

Summarize the dataset structure and notable quality observations. Use the findings to propose, but not execute, preprocessing decisions in a later phase.

## Cleaning decisions to be implemented later

- [ ]